In [1]:
import pandas as pd
from pathlib import Path

# ============================================================
# DISEASE KNOWLEDGE DATASET - PRODUCTION QUALITY CLEANING
# ============================================================

INPUT_FILE = "disease_knowledge_dataset_clean.csv"
OUTPUT_FILE = "disease_knowledge_dataset_cleaned.csv"

# Create output directory
Path("data/processed").mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# 1. Load dataset
# ------------------------------------------------------------

df = pd.read_csv(INPUT_FILE)

print("=" * 60)
print("DISEASE KNOWLEDGE DATASET CLEANING")
print("=" * 60)

print("Original shape:", df.shape)

# ------------------------------------------------------------
# 2. Clean column names
# ------------------------------------------------------------

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print("\nColumns:")
print(df.columns.tolist())

# ------------------------------------------------------------
# 3. Clean text columns
# ------------------------------------------------------------

text_columns = df.select_dtypes(
    include=["object", "string"]
).columns

for column in text_columns:

    df[column] = (
        df[column]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# ------------------------------------------------------------
# 4. Standardize common crop/country text
# ------------------------------------------------------------

# Standardize country if present
if "country" in df.columns:

    df["country"] = df["country"].replace({
        "IND": "India",
        "INDIA": "India",
        "india": "India"
    })

# Standardize crop capitalization if present
if "crop" in df.columns:

    df["crop"] = (
        df["crop"]
        .str.strip()
        .str.lower()
        .str.title()
    )

# Standardize disease names if present
if "disease" in df.columns:

    df["disease"] = (
        df["disease"]
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# ------------------------------------------------------------
# 5. Replace placeholder values with missing values
# ------------------------------------------------------------

placeholder_values = [
    "",
    " ",
    "na",
    "n/a",
    "none",
    "null",
    "unknown",
    "not available",
    "not_available"
]

for column in text_columns:

    df[column] = df[column].replace(
        placeholder_values,
        pd.NA
    )

# ------------------------------------------------------------
# 6. Remove completely empty rows
# ------------------------------------------------------------

rows_before_empty = len(df)

df = df.dropna(
    how="all"
)

empty_rows_removed = (
    rows_before_empty - len(df)
)

# ------------------------------------------------------------
# 7. Remove exact duplicate records
# ------------------------------------------------------------

rows_before_duplicates = len(df)

df = df.drop_duplicates()

df = df.reset_index(drop=True)

duplicates_removed = (
    rows_before_duplicates - len(df)
)

# ------------------------------------------------------------
# 8. Check important knowledge fields
# ------------------------------------------------------------

knowledge_columns = [
    "crop",
    "disease",
    "symptoms",
    "cause",
    "treatment",
    "management",
    "prevention"
]

existing_knowledge_columns = [
    column
    for column in knowledge_columns
    if column in df.columns
]

print("\nKnowledge columns found:")
print(existing_knowledge_columns)

# ------------------------------------------------------------
# 9. Validate empty knowledge fields
# ------------------------------------------------------------

if existing_knowledge_columns:

    print("\nMissing knowledge fields:")

    print(
        df[existing_knowledge_columns].isna().sum()
    )

# ------------------------------------------------------------
# 10. Remove rows without basic disease identity
# ------------------------------------------------------------

# Only apply this if both crop and disease columns exist.
# We don't want anonymous records entering the advisory system.

if "crop" in df.columns and "disease" in df.columns:

    before_identity_filter = len(df)

    df = df.dropna(
        subset=["crop", "disease"]
    )

    identity_rows_removed = (
        before_identity_filter - len(df)
    )

else:

    identity_rows_removed = 0

# ------------------------------------------------------------
# 11. Final validation
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CLEANING SUMMARY")
print("=" * 60)

print(
    "Original rows:",
    rows_before_empty
)

print(
    "Empty rows removed:",
    empty_rows_removed
)

print(
    "Duplicate rows removed:",
    duplicates_removed
)

print(
    "Rows without crop/disease identity removed:",
    identity_rows_removed
)

print(
    "Final rows:",
    len(df)
)

print("\nMissing values:")

print(
    df.isna().sum()
)

print("\nFinal columns:")

print(
    df.columns.tolist()
)

# ------------------------------------------------------------
# 12. Save cleaned dataset
# ------------------------------------------------------------

df.to_csv(
    OUTPUT_FILE,
    index=False
)

print("\n" + "=" * 60)
print("CLEANING COMPLETED SUCCESSFULLY")
print("=" * 60)

print(
    "Cleaned file saved to:",
    OUTPUT_FILE
)

DISEASE KNOWLEDGE DATASET CLEANING
Original shape: (26, 12)

Columns:
['crop', 'disease', 'pathogen_or_cause', 'symptoms', 'favorable_conditions', 'prevention', 'management', 'organic_management', 'chemical_management', 'severity', 'source', 'source_url']

Knowledge columns found:
['crop', 'disease', 'symptoms', 'management', 'prevention']

Missing knowledge fields:
crop          0
disease       0
symptoms      0
management    0
prevention    0
dtype: int64

CLEANING SUMMARY
Original rows: 26
Empty rows removed: 0
Duplicate rows removed: 0
Rows without crop/disease identity removed: 0
Final rows: 26

Missing values:
crop                    0
disease                 0
pathogen_or_cause       0
symptoms                0
favorable_conditions    0
prevention              0
management              0
organic_management      0
chemical_management     0
severity                0
source                  0
source_url              0
dtype: int64

Final columns:
['crop', 'disease', 'pathogen_or_ca